# 🏷️ Industrial Bottle & Packaging Label Scraper + Auto-Annotator
**Fully Autonomous Pipeline: Scrapes Real Industrial Bottle Labels & Auto-Annotates with EasyOCR (GPU)**

### Key Improvements:
* **PyTorch Native (EasyOCR):** Eliminates PaddlePaddle Python 3.13 crashes completely. Uses native PyTorch T4 GPU.
* **High-Intent Search Prompts:** Collects real commercial medicine bottles, cans, and jars.
* **Clean Downloader:** Suppresses crawler 403 warnings.
* **Standard COCO Format:** Outputs polygon annotations  matching our DBNet pipeline.
* **Visual Validation:** Displays detected polygon overlays directly in Colab.

> Make sure Colab runtime is set to **T4 GPU** ( ->  -> select ).

In [ ]:
# Cell 1: Install Dependencies (PyTorch-Native EasyOCR)
!pip install -q easyocr icrawler pillow shapely matplotlib opencv-python
print("✓ Dependencies installed successfully!")

In [ ]:
# Cell 2: Configure Targeted Industrial Search Queries
import os, json, shutil, logging
from PIL import Image

SEARCH_QUERIES = [
    "medicine bottle label close up photo real",
    "prescription pill bottle label printed text photo",
    "pharmaceutical cough syrup amber bottle label inspection",
    "beer bottle label macro close up photo",
    "wine bottle label front view photography real",
    "aluminum soda can label curved packaging printed text photo",
    "shampoo bottle label ingredients printed text photo",
    "cosmetic cream jar label packaging photo close up",
    "canned food label tin can packaging photo",
    "energy drink can nutrition label close up photo"
]

IMAGES_PER_QUERY = 25  # 10 queries x 25 = ~250 candidate images
RAW_DIR = "scraped_raw"
FINAL_IMG_DIR = "google_labels/train_images"
FINAL_ANN_DIR = "google_labels/annotations"

print(f"Configured {len(SEARCH_QUERIES)} targeted industrial search queries.")
print(f"Target collection: ~{len(SEARCH_QUERIES) * IMAGES_PER_QUERY} candidate images.")

In [ ]:
# Cell 3: Multi-Threaded Image Crawler (Silent Mode)
from icrawler.builtin import BingImageCrawler

# Suppress crawler network warning logs (403/502 from third-party sites)
logging.getLogger("downloader").setLevel(logging.CRITICAL)
logging.getLogger("parser").setLevel(logging.CRITICAL)
logging.getLogger("crawler").setLevel(logging.CRITICAL)

os.makedirs(RAW_DIR, exist_ok=True)

for i, query in enumerate(SEARCH_QUERIES, 1):
    print(f"[{i}/{len(SEARCH_QUERIES)}] Downloading: '{query}'")
    temp_folder = os.path.join(RAW_DIR, f"query_{i:02d}")
    os.makedirs(temp_folder, exist_ok=True)
    
    crawler = BingImageCrawler(storage={"root_dir": temp_folder}, log_level=logging.CRITICAL)
    crawler.crawl(keyword=query, max_num=IMAGES_PER_QUERY, min_size=(400, 400))

total_downloaded = sum(len(files) for _, _, files in os.walk(RAW_DIR))
print(f"Download complete! Successfully collected {total_downloaded} candidate images.")


In [ ]:
# Cell 4: Auto-Annotate using EasyOCR GPU into COCO Polygon Format
import easyocr
import torch

use_gpu = torch.cuda.is_available()
print(f"Initializing EasyOCR (GPU: {use_gpu})...")
reader = easyocr.Reader(["en"], gpu=use_gpu)

os.makedirs(FINAL_IMG_DIR, exist_ok=True)
os.makedirs(FINAL_ANN_DIR, exist_ok=True)

coco_images = []
coco_annotations = []
img_id = 1
ann_id = 1
rejected_no_text = 0

all_files = []
for root, _, files in os.walk(RAW_DIR):
    for f in sorted(files):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_files.append(os.path.join(root, f))

print(f"Scanning and annotating {len(all_files)} images on GPU...")

for idx, src_path in enumerate(all_files, 1):
    # Verify and load image
    try:
        with Image.open(src_path) as im:
            im_rgb = im.convert("RGB")
            w, h = im_rgb.size
            if w / h > 3.8 or h / w > 3.8:
                continue
    except Exception:
        continue

    # Run EasyOCR text detection
    try:
        results = reader.readtext(src_path)
    except Exception as e:
        print(f"  [Image {idx}] Error: {e}")
        continue

    # Filter: Must contain at least 2 text words with confidence > 0.15
    valid_detections = [item for item in results if item[2] >= 0.15]
    if len(valid_detections) < 2:
        rejected_no_text += 1
        continue

    # Save valid annotated bottle image
    new_fname = f"bottle_label_{img_id:04d}.jpg"
    dst_path = os.path.join(FINAL_IMG_DIR, new_fname)
    im_rgb.save(dst_path, "JPEG", quality=95)

    coco_images.append({
        "id": img_id,
        "file_name": new_fname,
        "width": w,
        "height": h,
        "source": "WebScraped_Industrial"
    })

    # Parse polygon bounding boxes
    for box, text_val, conf in valid_detections:
        # box: [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
        pts = []
        for pt in box:
            pts.extend([round(float(pt[0]), 2), round(float(pt[1]), 2)])

        xs = [pt[0] for pt in box]
        ys = [pt[1] for pt in box]
        min_x, max_x = min(xs), max(xs)
        min_y, max_y = min(ys), max(ys)
        bw = round(float(max_x - min_x), 2)
        bh = round(float(max_y - min_y), 2)
        area = round(bw * bh, 2)

        coco_annotations.append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": 1,
            "iscrowd": 0,
            "area": area,
            "bbox": [round(float(min_x), 2), round(float(min_y), 2), bw, bh],
            "segmentation": [pts],
            "transcription": str(text_val),
            "confidence": round(float(conf), 3)
        })
        ann_id += 1

    img_id += 1
    if img_id % 10 == 0:
        print(f"  ✓ Annotated {img_id-1} bottle labels ({ann_id-1} text regions detected)...")

# Save final COCO format JSON
out_json = os.path.join(FINAL_ANN_DIR, "train_google_labels.json")
with open(out_json, "w") as fp:
    json.dump({
        "images": coco_images,
        "annotations": coco_annotations,
        "categories": [{"id": 1, "name": "text"}]
    }, fp, indent=2)

print("=" * 60)
print("AUTO-ANNOTATION COMPLETE SUMMARY:")
print(f"  Valid Annotated Bottle Labels: {len(coco_images)}")
print(f"  Total Text Polygon Annotations: {len(coco_annotations)}")
print(f"  Rejected (Too few text regions): {rejected_no_text}")
print(f"  Saved COCO Annotations to:      {out_json}")
print("="*60)

In [ ]:
# Cell 5: Visual Validation (Inspect Auto-Annotations Directly in Colab)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

if len(coco_images) > 0:
    sample_count = min(6, len(coco_images))
    sample_imgs = coco_images[:sample_count]
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    img_to_anns = {}
    for a in coco_annotations:
        img_to_anns.setdefault(a["image_id"], []).append(a)
    
    for i, img_entry in enumerate(sample_imgs):
        img_path = os.path.join(FINAL_IMG_DIR, img_entry["file_name"])
        im = cv2.imread(img_path)
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(im)
        axes[i].set_title(f"{img_entry['file_name']} ({len(img_to_anns.get(img_entry['id'], []))} text regions)", fontsize=11)
        axes[i].axis("off")
        
        for ann in img_to_anns.get(img_entry["id"], []):
            seg = ann["segmentation"][0]
            poly_pts = [(seg[k], seg[k+1]) for k in range(0, len(seg), 2)]
            polygon = patches.Polygon(poly_pts, linewidth=2, edgecolor="#00FF00", facecolor="none")
            axes[i].add_patch(polygon)
    
    # Hide any unused subplots
    for j in range(sample_count, len(axes)):
        axes[j].axis("off")
        
    plt.suptitle("Auto-Annotation Visual Validation (Green = Detected Text Polygons)", fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("No images annotated to display.")

In [ ]:
# Cell 6: Package & Download Zip
from google.colab import files

zip_filename = "scraped_labels_coco"
if os.path.exists("google_labels") and len(os.listdir(FINAL_IMG_DIR)) > 0:
    shutil.make_archive(zip_filename, "zip", "google_labels")
    file_size_mb = os.path.getsize(f"{zip_filename}.zip") / (1024 * 1024)
    print(f"✓ Package created: {zip_filename}.zip ({file_size_mb:.2f} MB)")
    print("Initiating browser download...")
    files.download(f"{zip_filename}.zip")
else:
    print("Cannot create zip: No images found in google_labels/train_images!")